# DemoGrasp + PregraspPrior 实验说明

本文档只保留当前主线：`PregraspPrior` 先验、M/T/P field、yaw field/scorer 评估。旧 DGA 生成式 pregrasp bridge 已移除。

## 0. 环境准备

进入 DemoGrasp 根目录并激活环境。环境目录如果和本机不一致，替换为你当前实际使用的虚拟环境。

```bash
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine
source demograsp_new_coco_20260627_222202/bin/activate
```


## 1. M/T/P Rollout 采集与 Field 聚合

用途：采集 object-frame M/T/P 候选的 PPO rollout，并聚合成 `PregraspPrior/data/fields/ppo_random_mtp`。

输出：

```text
PregraspPrior/data/rollouts/ppo_random_mtp/
PregraspPrior/data/fields/ppo_random_mtp/
```

采集 rollout：

```bash
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

CUDA_VISIBLE_DEVICES=0 python3 -m build_Pregrasp_data.collect.collect_random_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  headless=true force_render=false num_envs=3200 \
  task.env.asset.multiObjectList="union_ycb_unidex/union_ycb_debugset.yaml" \
  +residual_checkpoint=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine/runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  +pregrasp_collect_max_batches=10 \
  +pregrasp_collect_mtp_mode=cycle \
  +pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_random_mtp
```

聚合 field：

```bash
python3 -m build_Pregrasp_data.aggregate.build_field_store_from_rollouts \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_random_mtp \
  --output-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_random_mtp \
  --tilt-angles 0,15,30 \
  --pitch-angles 0,15,30 \
  --overwrite
```


## 2. M/T/P Field 引导评估

用途：读取 `ppo_random_mtp` field，用 field 选择 M/T/P 初始化，然后跑 SE residual PPO。

### 2.1 完整 M/T/P 初始化

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_pregrasp_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  +pregrasp_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_random_mtp \
  +pregrasp_field_tilt_angles="[0,15,30]" \
  +pregrasp_field_pitch_angles="[0,15,30]" \
  +pregrasp_field_topk=1 \
  +pregrasp_field_selection=cycle \
  +pregrasp_collect_max_batches=10 \
  +pregrasp_collect_overwrite_root=True \
  +pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_field_guided_eval
```

### 2.2 只使用 M 方向初始化

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_pregrasp_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  +pregrasp_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_random_mtp \
  +pregrasp_field_control=m \
  +pregrasp_field_tilt_angles="[0,15,30]" \
  +pregrasp_field_pitch_angles="[0,15,30]" \
  +pregrasp_field_topk=1 \
  +pregrasp_field_selection=cycle \
  +pregrasp_collect_max_batches=10 \
  +pregrasp_collect_overwrite_root=True \
  +pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_field_guided_m_only
```


## 3. 扩展 Arc 可视化/检查

用途：使用 `ppo_m642` field 做 expanded training arc 检查。这个命令偏调试和可视化，不是最小闭环。

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_pregrasp_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=false force_render=true \
  num_envs=32 \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  +pregrasp_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_m642 \
  +pregrasp_field_control=expanded_training_arc \
  +pregrasp_field_expanded_step_deg=5 \
  +pregrasp_field_selection=random \
  +pregrasp_collect_subdivision=3 \
  +pregrasp_collect_max_batches=10 \
  +pregrasp_collect_overwrite_root=True \
  +pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_field_expanded_arc_view \
  +pregrasp_eval_log_interval=0
```


## 4. Yaw 候选直接评估

用途：不使用已聚合 field，直接按 yaw/tilt/pitch 候选循环初始化，用来检查 yaw 采样和 canonical observation 设置。

### 4.1 单物体可视化检查

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=false force_render=true \
  num_envs=24 \
  task.env.asset.multiObject=false \
  +task.env.asset.objectAssetFile=union_ycb_unidex/urdf/065-f_cups.urdf \
  task.env.resetRandomRot=fixed \
  task.env.resetPositionRange='[[0.5,0.5],[-0.1,-0.1],[0.11,0.11]]' \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=15 \
  --pitch_angles=15 \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=hand \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_eval_log_interval=0
```

### 4.2 多物体批量评估

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=3200 \
  task.env.asset.multiObject=true \
  task.env.asset.multiObjectList="union_ycb_unidex/union_ycb_debugset.yaml" \
  task.env.resetRandomRot=z \
  task.env.resetPositionRange='[[0.4,0.65],[-0.2,0.05],[0.10,0.12]]' \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=0,15,30 \
  --pitch_angles=0,15,30 \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=hand \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_eval_log_interval=0
```


## 5. Yaw Field 小规模 Debug 闭环

用途：单物体、小 env 数，快速检查 yaw rollout 采集、聚合、field_top1 评估三步是否连通。

采集：

```bash
CUDA_VISIBLE_DEVICES=0 python3 -m build_Pregrasp_data.yaw_pregrasp.collect.collect_yaw_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  headless=true force_render=false num_envs=108 \
  task.env.asset.multiObject=false \
  +task.env.asset.objectAssetFile=union_ycb_unidex/urdf/065-f_cups.urdf \
  task.env.resetRandomRot=fixed \
  task.env.resetPositionRange='[[0.5,0.5],[-0.1,-0.1],[0.11,0.11]]' \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=0,15,30 \
  --pitch_angles=0,15,30 \
  +pregrasp_yaw_sampling=cycle \
  +yaw_pregrasp_collect_max_batches=1 \
  +yaw_pregrasp_collect_overwrite_root=True \
  +yaw_pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/yaw_debug \
  +pregrasp_eval_log_interval=0
```

聚合：

```bash
python3 -m build_Pregrasp_data.yaw_pregrasp.aggregate.build_yaw_field_store \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/yaw_debug \
  --field-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/yaw_debug
```

评估：

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=false force_render=true \
  num_envs=20 \
  task.env.asset.multiObject=false \
  +task.env.asset.objectAssetFile=union_ycb_unidex/urdf/065-f_cups.urdf \
  task.env.resetRandomRot=fixed \
  task.env.resetPositionRange='[[0.5,0.5],[-0.1,-0.1],[0.11,0.11]]' \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=0,15,30 \
  --pitch_angles=0,15,30 \
  +pregrasp_yaw_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/yaw_debug \
  +pregrasp_yaw_sampling=field_top1 \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=hand \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_eval_log_interval=0
```


## 6. Yaw Field 多物体主实验

用途：多物体 `pca_short` yaw12 设置。采集、聚合、评估都使用同一组目录：`yaw_pca_short_yaw12_scene`。

采集：

```bash
CUDA_VISIBLE_DEVICES=0 python3 -m build_Pregrasp_data.yaw_pregrasp.collect.collect_yaw_pregrasp_rollouts \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  headless=true force_render=false num_envs=3200 \
  task.env.asset.multiObjectList="union_ycb_unidex/union_ycb_debugset.yaml" \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=0 \
  --pitch_angles=0 \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_sampling=cycle \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +yaw_pregrasp_collect_max_batches=5 \
  +yaw_pregrasp_collect_overwrite_root=True \
  +yaw_pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/yaw_pca_short_yaw12_scene \
  +pregrasp_eval_log_interval=0
```

聚合：

```bash
python3 -m build_Pregrasp_data.yaw_pregrasp.aggregate.build_yaw_field_store \
  --rollout-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/yaw_pca_short_yaw12_scene \
  --field-root /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/yaw_pca_short_yaw12_scene
```

评估：

```bash
CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_yaw_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=false force_render=true \
  num_envs=32 \
  task.env.asset.multiObjectList="union_ycb_unidex/union_ycb_debugset.yaml" \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  --yaw_angles=0,30,60,90,120,150,180,210,240,270,300,330 \
  --tilt_angles=0 \
  --pitch_angles=0 \
  +pregrasp_yaw_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/yaw_pca_short_yaw12_scene \
  +pregrasp_yaw_frame=pca_short \
  +pregrasp_yaw_sampling=field_top1 \
  +pregrasp_yaw_canonical_obs=True \
  +pregrasp_yaw_canonical_obs_scope=scene \
  +pregrasp_yaw_direct_base_qpos=True \
  +pregrasp_yaw_compare_base_rounds=1 \
  +pregrasp_yaw_compare_field_rounds=500 \
  +pregrasp_yaw_compare_base_mode=all \
  +pregrasp_yaw_compare_base_sampling=random \
  +pregrasp_eval_log_interval=0
```

等价脚本入口：

```bash
BATCHES=10 CUDA_VISIBLE_DEVICES=0 bash /mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/scripts/collect_pca_short_yaw12_tilt3_pitch3_scene.sh
```


## 7. 最小闭环测试

建议每次大规模运行前先跑这一段。它只检查 field store 是否能读，并用 `num_envs=4` 跑一次 heatmap 引导评估。

```bash
cd /mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine

PYTHONPATH=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/src:/mnt/data1/zju/zhangziling/AAAI2027_Grasp/DemoGrasp_mine python3 - <<'PY'
from oc_pregrasp.field.field_store import FieldStore
root = '/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_random_mtp'
store = FieldStore(root)
print('field entries:', len(store.entries))
print('first object:', store.entries[0]['object_name'] if store.entries else 'empty')
PY

CUDA_VISIBLE_DEVICES=0 python3 pregrasp_eval/eval_pregrasp_field_SE_residual.py \
  task=grasp train=PPOOneStep hand=new_sr_hand_simple \
  test=True headless=true force_render=false \
  num_envs=4 \
  checkpoint=runs_ppo/2026-06-22_07-59-10_new_sr_hand_simple_se_tilt_pitch_0-15-30_pitch_0-15-30/model_15000.pt \
  +pregrasp_field_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/fields/ppo_random_mtp \
  +pregrasp_field_tilt_angles="[0,15,30]" \
  +pregrasp_field_pitch_angles="[0,15,30]" \
  +pregrasp_field_topk=1 \
  +pregrasp_field_selection=cycle \
  +pregrasp_collect_max_batches=1 \
  +pregrasp_collect_overwrite_root=True \
  +pregrasp_collect_rollout_root=/mnt/data1/zju/zhangziling/AAAI2027_Grasp/PregraspPrior/data/rollouts/ppo_field_guided_smoke
```
